# 03 — Cortex Agent

This notebook creates:
1. **Semantic View** — business-friendly model over the GOLD layer
2. **Cortex Agent** — AI-powered fleet intelligence assistant


In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA GOLD;
USE WAREHOUSE HYDRAB_HOL_WH;

## Semantic View

The Semantic View defines a business-friendly model for Cortex Analyst.

In [ ]:
-- Create Semantic View using SQL DDL syntax
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA GOLD;

CREATE OR REPLACE SEMANTIC VIEW FLEET_SEMANTIC

  TABLES (
    vehicles AS DIM_VEHICLE
      COMMENT = 'Vehicle master data',
    telemetry AS FCT_LATEST_TELEMETRY
      COMMENT = 'Latest telemetry readings per vehicle'
  )

  RELATIONSHIPS (
    telemetry_to_vehicles AS
      telemetry (vin) REFERENCES vehicles (vin)
  )

  DIMENSIONS (
    vehicles.dim_vin AS vin
      COMMENT = 'Vehicle Identification Number',
    vehicles.dim_model AS model
      COMMENT = 'Bus model name',
    vehicles.dim_status AS status
      COMMENT = 'Vehicle lifecycle status',
    vehicles.dim_delivery_status AS delivery_status
      COMMENT = 'Delivery pipeline status',
    telemetry.dim_customer AS customer_name
      COMMENT = 'Operating customer'
  )

  METRICS (
    vehicles.total_vehicles AS COUNT(DISTINCT vin)
      COMMENT = 'Total number of vehicles',
    vehicles.deployed_count AS COUNT_IF(status = 'Deployed')
      COMMENT = 'Deployed vehicles',
    telemetry.avg_battery_soc AS AVG(battery_soc)
      COMMENT = 'Average battery state of charge',
    telemetry.avg_speed AS AVG(speed_kmh)
      COMMENT = 'Average speed km/h',
    telemetry.low_battery_count AS COUNT_IF(battery_soc < 20)
      COMMENT = 'Vehicles with SOC below 20 percent'
  )

  COMMENT = 'HydraB fleet analytics - vehicles and telemetry';

## Cortex Agent

The Agent combines structured analytics (via Semantic View) with natural language Q&A.

In [ ]:
CREATE OR REPLACE CORTEX AGENT GOLD.FLEET_AGENT
  COMMENT = 'HydraB Fleet Intelligence Agent'
AS
  USING (
    CORTEX_ANALYST(
      SEMANTIC_VIEW => 'GOLD.FLEET_SEMANTIC'
    )
  );

In [ ]:
-- Test the agent
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'GOLD.FLEET_AGENT',
  'Which bus models have the most defects and what is their average repair cost?'
) AS agent_response;

## Gold Layer Complete!

You now have:
- Auto-refreshing Dynamic Tables in SILVER
- Star schema views in GOLD
- A Semantic View for natural language analytics
- A Cortex Agent you can ask questions

**Next:** Open `04_deploy_dashboard.ipynb` to deploy the React dashboard.